# Cyclistic Bike Share Analysis
## 02 — Data Cleaning and Feature Engineering

### Study Period
July 2025 to June 2026

### Purpose
This notebook cleans the combined Cyclistic dataset produced during the data-preparation stage.

The cleaning process includes:

- restricting records to the defined study period;
- investigating duplicate ride IDs;
- removing confirmed duplicate records;
- validating ride durations;
- examining missing values;
- creating analytical features such as ride length, weekday, month, and hour;
- exporting the cleaned dataset for exploratory analysis.

In [5]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [6]:
cwd = Path.cwd()

if (cwd / "03_cleaned_data").exists():
    project_path = cwd
elif (cwd.parent / "03_cleaned_data").exists():
    project_path = cwd.parent
else:
    raise FileNotFoundError(
        "Could not locate the Cyclistic project folder."
    )

cleaned_data_path = project_path / "03_cleaned_data"
intermediate_path = cleaned_data_path / "intermediate"
documentation_path = project_path / "06_documentation"

print("Project folder:", project_path)
print("Intermediate data:", intermediate_path)

Project folder: c:\Users\OLUWATOSIN OLUWASEUN\Downloads\Cyclistic_Bike_Share_Case_Study
Intermediate data: c:\Users\OLUWATOSIN OLUWASEUN\Downloads\Cyclistic_Bike_Share_Case_Study\03_cleaned_data\intermediate


In [7]:
combined_file = (
    intermediate_path /
    "cyclistic_202507_202606_combined_raw.parquet"
)

df = pd.read_parquet(combined_file)

print(f"Rows loaded: {len(df):,}")
print(f"Columns: {df.shape[1]}")

Rows loaded: 5,932,349
Columns: 14


In [8]:
print("Starting validation:")

print(f"Total rows: {len(df):,}")
print(f"Unique ride IDs: {df['ride_id'].nunique():,}")
print(f"Duplicate rows by ride_id: {df['ride_id'].duplicated().sum():,}")
print(f"Missing started_at: {df['started_at'].isna().sum():,}")
print(f"Missing ended_at: {df['ended_at'].isna().sum():,}")

Starting validation:
Total rows: 5,932,349
Unique ride IDs: 5,932,314
Duplicate rows by ride_id: 35
Missing started_at: 0
Missing ended_at: 0


In [9]:
study_start = pd.Timestamp("2025-07-01")
study_end = pd.Timestamp("2026-07-01")

in_study_period = (
    (df["started_at"] >= study_start) &
    (df["started_at"] < study_end)
)

print(
    "Rows outside study period:",
    (~in_study_period).sum()
)

Rows outside study period: 102


In [10]:
df.loc[
    ~in_study_period,
    [
        "ride_id",
        "started_at",
        "ended_at",
        "source_file"
    ]
].sort_values("started_at").head(20)

,ride_id,started_at,ended_at,source_file
760033,495F0D6620B0AF6E,2025-06-29 23:20:58.472,2025-07-01 00:20:51.012,202507-divvy-tripdata.csv
22364,927F666723964196,2025-06-30 01:36:37.591,2025-07-01 02:36:34.855,202507-divvy-tripdata.csv
760776,39253618A91EB6C5,2025-06-30 02:44:49.024,2025-07-01 03:44:45.178,202507-divvy-tripdata.csv
55462,A6B67B2740E413CE,2025-06-30 03:34:37.028,2025-07-01 04:34:32.647,202507-divvy-tripdata.csv
55461,DCD12F835C2FE9B2,2025-06-30 03:34:47.137,2025-07-01 04:34:43.969,202507-divvy-tripdata.csv
713764,F4B89C59B8D95D24,2025-06-30 11:00:42.273,2025-07-01 12:00:36.794,202507-divvy-tripdata.csv
734479,E1F45A3DC15EEFDE,2025-06-30 11:50:04.626,2025-07-01 12:49:57.918,202507-divvy-tripdata.csv
22938,A316814421937BC7,2025-06-30 11:54:21.498,2025-07-01 12:54:01.047,202507-divvy-tripdata.csv
716409,99EE23852978438B,2025-06-30 12:56:27.241,2025-07-01 13:56:19.442,202507-divvy-tripdata.csv
713763,59FA0A3BE09FD522,2025-06-30 13:02:15.819,2025-07-01 14:02:12.432,202507-divvy-tripdata.csv


In [11]:
df_clean = df.loc[in_study_period].copy()

print(f"Rows before filtering: {len(df):,}")
print(f"Rows removed: {(~in_study_period).sum():,}")
print(f"Rows remaining: {len(df_clean):,}")

Rows before filtering: 5,932,349
Rows removed: 102
Rows remaining: 5,932,247


In [12]:
duplicate_records = df_clean[
    df_clean["ride_id"].duplicated(keep=False)
].sort_values("ride_id")

print(
    "Rows belonging to duplicated ride IDs:",
    len(duplicate_records)
)

print(
    "Unique duplicated ride IDs:",
    duplicate_records["ride_id"].nunique()
)

Rows belonging to duplicated ride IDs: 70
Unique duplicated ride IDs: 35


In [13]:
duplicate_records.head(30)

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual,source_file
4311956,03C5375398BE0649,electric_bike,2026-04-30 23:46:36.039,2026-05-01 00:04:48.512,<NA>,<NA>,Clark St & Arlington Pl,CHI02109,41.900000,-87.630000,41.926940,-87.641480,member,202604-divvy-tripdata.csv
5052810,03C5375398BE0649,electric_bike,2026-04-30 23:46:36.039,2026-05-01 00:04:48.512,<NA>,<NA>,Clark St & Arlington Pl,CHI02109,41.900000,-87.630000,41.926940,-87.641480,member,202605-divvy-tripdata.csv
4433288,0612B2A21BA5E988,classic_bike,2026-04-30 15:56:11.305,2026-05-01 16:56:08.148,Fulton Market,CHI01880,<NA>,<NA>,41.886871,-87.648089,NaN,NaN,member,202604-divvy-tripdata.csv
4775869,0612B2A21BA5E988,classic_bike,2026-04-30 15:56:11.305,2026-05-01 16:56:08.148,Fulton Market,CHI01880,<NA>,<NA>,41.886871,-87.648089,NaN,NaN,member,202605-divvy-tripdata.csv
4252515,10117825A92C182D,classic_bike,2026-04-30 23:56:24.321,2026-05-01 00:01:51.860,Peoria St & Kinzie St,CHI02098,Ogden Ave & Chicago Ave,CHI00275,41.888960,-87.649730,41.896362,-87.654061,casual,202604-divvy-tripdata.csv
4667877,10117825A92C182D,classic_bike,2026-04-30 23:56:24.321,2026-05-01 00:01:51.860,Peoria St & Kinzie St,CHI02098,Ogden Ave & Chicago Ave,CHI00275,41.888960,-87.649730,41.896362,-87.654061,casual,202605-divvy-tripdata.csv
4184705,15521787039F37E5,classic_bike,2026-04-30 23:54:44.934,2026-05-01 00:05:45.575,Michigan Ave & 11th St,CHI02025,DuSable Lake Shore Dr & Monroe St,CHI00374,41.869590,-87.623961,41.880958,-87.616743,member,202604-divvy-tripdata.csv
4934119,15521787039F37E5,classic_bike,2026-04-30 23:54:44.934,2026-05-01 00:05:45.575,Michigan Ave & 11th St,CHI02025,DuSable Lake Shore Dr & Monroe St,CHI00374,41.869590,-87.623961,41.880958,-87.616743,member,202605-divvy-tripdata.csv
4443628,1DB04E901469FB2F,electric_bike,2026-04-30 23:54:09.726,2026-05-01 00:11:09.709,Ravenswood Ave & Irving Park Rd,CHI00309,<NA>,<NA>,41.954690,-87.673930,41.970000,-87.670000,casual,202604-divvy-tripdata.csv
5132559,1DB04E901469FB2F,electric_bike,2026-04-30 23:54:09.726,2026-05-01 00:11:09.709,Ravenswood Ave & Irving Park Rd,CHI00309,<NA>,<NA>,41.954690,-87.673930,41.970000,-87.670000,casual,202605-divvy-tripdata.csv


In [14]:
study_start = pd.Timestamp("2025-07-01")
study_end = pd.Timestamp("2026-07-01")

in_study_period = (
    (df["started_at"] >= study_start) &
    (df["started_at"] < study_end)
)

print("Rows before filtering:", f"{len(df):,}")
print("Rows outside study period:", f"{(~in_study_period).sum():,}")

df_clean = df.loc[in_study_period].copy()

print("Rows after filtering:", f"{len(df_clean):,}")

Rows before filtering: 5,932,349
Rows outside study period: 102
Rows after filtering: 5,932,247


In [15]:
duplicate_records = df_clean[
    df_clean["ride_id"].duplicated(keep=False)
].sort_values("ride_id")

print(
    "Rows belonging to duplicated ride IDs:",
    len(duplicate_records)
)

print(
    "Distinct duplicated ride IDs:",
    duplicate_records["ride_id"].nunique()
)

duplicate_records.head(20)

Rows belonging to duplicated ride IDs: 70
Distinct duplicated ride IDs: 35


,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual,source_file
4311956,03C5375398BE0649,electric_bike,2026-04-30 23:46:36.039,2026-05-01 00:04:48.512,<NA>,<NA>,Clark St & Arlington Pl,CHI02109,41.900000,-87.630000,41.926940,-87.641480,member,202604-divvy-tripdata.csv
5052810,03C5375398BE0649,electric_bike,2026-04-30 23:46:36.039,2026-05-01 00:04:48.512,<NA>,<NA>,Clark St & Arlington Pl,CHI02109,41.900000,-87.630000,41.926940,-87.641480,member,202605-divvy-tripdata.csv
4433288,0612B2A21BA5E988,classic_bike,2026-04-30 15:56:11.305,2026-05-01 16:56:08.148,Fulton Market,CHI01880,<NA>,<NA>,41.886871,-87.648089,NaN,NaN,member,202604-divvy-tripdata.csv
4775869,0612B2A21BA5E988,classic_bike,2026-04-30 15:56:11.305,2026-05-01 16:56:08.148,Fulton Market,CHI01880,<NA>,<NA>,41.886871,-87.648089,NaN,NaN,member,202605-divvy-tripdata.csv
4252515,10117825A92C182D,classic_bike,2026-04-30 23:56:24.321,2026-05-01 00:01:51.860,Peoria St & Kinzie St,CHI02098,Ogden Ave & Chicago Ave,CHI00275,41.888960,-87.649730,41.896362,-87.654061,casual,202604-divvy-tripdata.csv
4667877,10117825A92C182D,classic_bike,2026-04-30 23:56:24.321,2026-05-01 00:01:51.860,Peoria St & Kinzie St,CHI02098,Ogden Ave & Chicago Ave,CHI00275,41.888960,-87.649730,41.896362,-87.654061,casual,202605-divvy-tripdata.csv
4184705,15521787039F37E5,classic_bike,2026-04-30 23:54:44.934,2026-05-01 00:05:45.575,Michigan Ave & 11th St,CHI02025,DuSable Lake Shore Dr & Monroe St,CHI00374,41.869590,-87.623961,41.880958,-87.616743,member,202604-divvy-tripdata.csv
4934119,15521787039F37E5,classic_bike,2026-04-30 23:54:44.934,2026-05-01 00:05:45.575,Michigan Ave & 11th St,CHI02025,DuSable Lake Shore Dr & Monroe St,CHI00374,41.869590,-87.623961,41.880958,-87.616743,member,202605-divvy-tripdata.csv
4443628,1DB04E901469FB2F,electric_bike,2026-04-30 23:54:09.726,2026-05-01 00:11:09.709,Ravenswood Ave & Irving Park Rd,CHI00309,<NA>,<NA>,41.954690,-87.673930,41.970000,-87.670000,casual,202604-divvy-tripdata.csv
5132559,1DB04E901469FB2F,electric_bike,2026-04-30 23:54:09.726,2026-05-01 00:11:09.709,Ravenswood Ave & Irving Park Rd,CHI00309,<NA>,<NA>,41.954690,-87.673930,41.970000,-87.670000,casual,202605-divvy-tripdata.csv


In [16]:
exact_duplicate_rows = df_clean.duplicated().sum()

print(
    "Completely identical duplicate rows:",
    exact_duplicate_rows
)

Completely identical duplicate rows: 0


In [17]:
duplicate_id_counts = (
    df_clean["ride_id"]
    .value_counts()
    .loc[lambda x: x > 1]
)

duplicate_id_counts.head(20)

ride_id
D582320F33836B82    2
D28B306F00821227    2
E923A7ACDD4A9293    2
2038F30519CBC7E5    2
5C56F4FB55D5B5F5    2
41E2750A57535CF7    2
15521787039F37E5    2
4EBCC6DF233878B6    2
EAB7C279D822C763    2
5232F2CB3D297D55    2
82EDB771CE5282E8    2
C9050956F58561AD    2
B046AA82A096C6FA    2
3D8B4B1B10438A9F    2
10117825A92C182D    2
78D590AAA74CAE97    2
796A83D45529C7D8    2
03C5375398BE0649    2
D45C58C1A07DDA84    2
C85BE84CF8D18F6D    2
Name: count, dtype: int64[pyarrow]

In [18]:
duplicate_comparison = (
    duplicate_records
    .groupby("ride_id")
    .nunique(dropna=False)
)

conflicting_duplicates = duplicate_comparison[
    (duplicate_comparison > 1).any(axis=1)
]

print(
    "Duplicated ride IDs with conflicting values:",
    len(conflicting_duplicates)
)

Duplicated ride IDs with conflicting values: 35


In [19]:
# Original Divvy columns only — exclude our added source_file field
comparison_columns = [
    col for col in df_clean.columns
    if col != "source_file"
]

duplicate_comparison_actual = (
    duplicate_records[comparison_columns]
    .groupby("ride_id")
    .nunique(dropna=False)
)

true_conflicting_duplicates = duplicate_comparison_actual[
    (duplicate_comparison_actual > 1).any(axis=1)
]

print(
    "Duplicated ride IDs with genuine conflicting trip data:",
    len(true_conflicting_duplicates)
)

Duplicated ride IDs with genuine conflicting trip data: 0


In [20]:
duplicate_sources = (
    duplicate_records
    .groupby("ride_id")["source_file"]
    .agg(list)
)

duplicate_sources.head(20)

ride_id
03C5375398BE0649    [202604-divvy-tripdata.csv, 202605-divvy-tripd...
0612B2A21BA5E988    [202604-divvy-tripdata.csv, 202605-divvy-tripd...
10117825A92C182D    [202604-divvy-tripdata.csv, 202605-divvy-tripd...
15521787039F37E5    [202604-divvy-tripdata.csv, 202605-divvy-tripd...
1DB04E901469FB2F    [202604-divvy-tripdata.csv, 202605-divvy-tripd...
2034B29BDC8E2559    [202604-divvy-tripdata.csv, 202605-divvy-tripd...
2038F30519CBC7E5    [202604-divvy-tripdata.csv, 202605-divvy-tripd...
3767A2E3310A5B0B    [202604-divvy-tripdata.csv, 202605-divvy-tripd...
3D8B4B1B10438A9F    [202604-divvy-tripdata.csv, 202605-divvy-tripd...
41E2750A57535CF7    [202604-divvy-tripdata.csv, 202605-divvy-tripd...
437C637BFCE893DE    [202604-divvy-tripdata.csv, 202605-divvy-tripd...
47B00C2CFA51BED8    [202604-divvy-tripdata.csv, 202605-divvy-tripd...
4B93DFFDA6B08DE5    [202604-divvy-tripdata.csv, 202605-divvy-tripd...
4EBCC6DF233878B6    [202604-divvy-tripdata.csv, 202605-divvy-tripd...
5232F2CB3D29

In [21]:
duplicate_without_source = duplicate_records.drop(
    columns="source_file"
)

exact_duplicate_trip_rows = (
    duplicate_without_source
    .duplicated()
    .sum()
)

print(
    "Duplicate rows identical across original Divvy fields:",
    exact_duplicate_trip_rows
)

Duplicate rows identical across original Divvy fields: 35


In [22]:
rows_before_duplicates = len(df_clean)

df_clean = (
    df_clean
    .drop_duplicates(
        subset="ride_id",
        keep="first"
    )
    .copy()
)

rows_removed_duplicates = (
    rows_before_duplicates - len(df_clean)
)

print(
    f"Duplicate ride records removed: "
    f"{rows_removed_duplicates:,}"
)

print(
    f"Rows remaining: "
    f"{len(df_clean):,}"
)

Duplicate ride records removed: 35
Rows remaining: 5,932,212


In [23]:
rows_before_duplicates = len(df_clean)

df_clean = (
    df_clean
    .drop_duplicates(
        subset="ride_id",
        keep="first"
    )
    .copy()
)

rows_removed_duplicates = (
    rows_before_duplicates - len(df_clean)
)

print(
    f"Duplicate ride records removed: "
    f"{rows_removed_duplicates:,}"
)

print(
    f"Rows remaining: "
    f"{len(df_clean):,}"
)

Duplicate ride records removed: 0
Rows remaining: 5,932,212


In [24]:
print(f"Current rows: {len(df_clean):,}")
print(f"Unique ride IDs: {df_clean['ride_id'].nunique():,}")
print(
    f"Remaining duplicate ride IDs: "
    f"{df_clean['ride_id'].duplicated().sum():,}"
)

Current rows: 5,932,212
Unique ride IDs: 5,932,212
Remaining duplicate ride IDs: 0


### Duplicate Ride IDs

Initial validation identified 35 additional records with duplicated `ride_id` values.

The duplicate records were investigated before removal. After restricting the dataset to the defined study period, one record was retained for each duplicated `ride_id`.

A total of 35 duplicate records were removed.

Rows after duplicate removal: 5,932,212.

In [25]:
df_clean["ride_length_minutes"] = (
    df_clean["ended_at"] - df_clean["started_at"]
).dt.total_seconds() / 60

df_clean[
    [
        "started_at",
        "ended_at",
        "ride_length_minutes"
    ]
].head()

,started_at,ended_at,ride_length_minutes
0,2025-07-05 17:15:08.456,2025-07-05 17:25:47.079,10.643717
1,2025-07-01 13:57:38.878,2025-07-01 14:06:35.780,8.948367
2,2025-07-31 16:49:28.142,2025-07-31 17:15:28.999,26.014283
3,2025-07-17 09:36:21.058,2025-07-17 09:46:54.706,10.560800
4,2025-07-02 18:43:45.213,2025-07-02 18:57:06.687,13.357900


In [26]:
df_clean["ride_length_minutes"].describe()

count    5.932212e+06
mean     1.576184e+01
std      5.337470e+01
min     -5.479480e+01
25%      5.359183e+00
50%      9.368150e+00
75%      1.647012e+01
max      1.559950e+03
Name: ride_length_minutes, dtype: float64

In [27]:
non_positive_rides = (
    df_clean["ride_length_minutes"] <= 0
).sum()

print(
    f"Rides with duration <= 0 minutes: "
    f"{non_positive_rides:,}"
)

print(
    "Shortest ride:",
    df_clean["ride_length_minutes"].min()
)

print(
    "Longest ride:",
    df_clean["ride_length_minutes"].max()
)

Rides with duration <= 0 minutes: 29
Shortest ride: -54.7948
Longest ride: 1559.9501833333334


In [28]:
df_clean.loc[
    df_clean["ride_length_minutes"] <= 0,
    [
        "ride_id",
        "started_at",
        "ended_at",
        "ride_length_minutes",
        "member_casual",
        "source_file"
    ]
].head(20)

,ride_id,started_at,ended_at,ride_length_minutes,member_casual,source_file
2939528,5D010AFEA6850513,2025-11-02 01:52:37.475,2025-11-02 01:13:24.728,-39.212450,member,202511-divvy-tripdata.csv
2941076,D1E5316AD88ECD45,2025-11-02 01:50:39.702,2025-11-02 01:14:04.164,-36.592300,casual,202511-divvy-tripdata.csv
2943070,3AF2F8908C9386F8,2025-11-02 01:57:57.512,2025-11-02 01:33:52.225,-24.088117,member,202511-divvy-tripdata.csv
2957966,19386939ECD81B33,2025-11-02 01:55:34.399,2025-11-02 01:21:31.873,-34.042100,member,202511-divvy-tripdata.csv
2966334,083534D28DA37F72,2025-11-02 01:17:57.001,2025-11-02 01:05:27.752,-12.487483,member,202511-divvy-tripdata.csv
3010919,4A659F1BB40EDBF7,2025-11-02 01:55:38.177,2025-11-02 01:04:35.315,-51.047700,casual,202511-divvy-tripdata.csv
3036472,DA5386E5759667EC,2025-11-02 01:51:31.704,2025-11-02 01:05:47.339,-45.739417,member,202511-divvy-tripdata.csv
3041930,985A3462DAACFFCF,2025-11-02 01:59:07.835,2025-11-02 01:05:19.214,-53.810350,member,202511-divvy-tripdata.csv
3061853,434E3D11ABA96891,2025-11-02 01:59:40.156,2025-11-02 01:08:51.303,-50.814217,casual,202511-divvy-tripdata.csv
3071671,9288498F5630E3AD,2025-11-02 01:49:48.360,2025-11-02 01:02:40.976,-47.123067,casual,202511-divvy-tripdata.csv


In [29]:
duration_summary = df_clean["ride_length_minutes"].describe(
    percentiles=[0.001, 0.01, 0.25, 0.5, 0.75, 0.99, 0.999]
)

duration_summary

count    5.932212e+06
mean     1.576184e+01
std      5.337470e+01
min     -5.479480e+01
0.1%     2.548333e-02
1%       2.414833e-01
25%      5.359183e+00
50%      9.368150e+00
75%      1.647012e+01
99%      8.709339e+01
99.9%    1.267057e+03
max      1.559950e+03
Name: ride_length_minutes, dtype: float64

In [30]:
print(
    "Rides <= 0 minutes:",
    f"{(df_clean['ride_length_minutes'] <= 0).sum():,}"
)

print(
    "Rides < 1 minute:",
    f"{(df_clean['ride_length_minutes'] < 1).sum():,}"
)

print(
    "Rides > 60 minutes:",
    f"{(df_clean['ride_length_minutes'] > 60).sum():,}"
)

print(
    "Rides > 24 hours:",
    f"{(df_clean['ride_length_minutes'] > 1440).sum():,}"
)

Rides <= 0 minutes: 29
Rides < 1 minute: 162,246
Rides > 60 minutes: 122,529
Rides > 24 hours: 5,513


In [31]:
df_clean[
    [
        "ride_id",
        "started_at",
        "ended_at",
        "ride_length_minutes",
        "rideable_type",
        "member_casual",
        "source_file"
    ]
].sort_values(
    "ride_length_minutes",
    ascending=False
).head(20)

,ride_id,started_at,ended_at,ride_length_minutes,rideable_type,member_casual,source_file
3818349,C82393BDD3070D5D,2026-03-07 16:07:09.988,2026-03-08 18:07:06.999,1559.950183,classic_bike,casual,202603-divvy-tripdata.csv
3895510,250556580DD308BE,2026-03-07 23:51:38.812,2026-03-09 01:51:33.033,1559.903683,classic_bike,member,202603-divvy-tripdata.csv
3892036,266FF1F3D3203623,2026-03-07 12:29:39.955,2026-03-08 14:29:33.587,1559.893867,classic_bike,casual,202603-divvy-tripdata.csv
3896432,F2DADD2100A676E7,2026-03-07 09:43:11.203,2026-03-08 11:43:04.158,1559.882583,classic_bike,casual,202603-divvy-tripdata.csv
3818481,EB04F9C062AAAF6A,2026-03-07 09:33:18.048,2026-03-08 11:33:09.975,1559.865450,classic_bike,casual,202603-divvy-tripdata.csv
3890261,7CFD1B6B5319BB28,2026-03-07 12:20:32.025,2026-03-08 14:20:23.251,1559.853767,classic_bike,casual,202603-divvy-tripdata.csv
3890264,B35EFB4539BB4B2B,2026-03-07 02:41:16.669,2026-03-08 04:41:03.312,1559.777383,classic_bike,casual,202603-divvy-tripdata.csv
3934526,13769591C639EB0C,2026-03-07 12:45:52.881,2026-03-08 14:45:33.445,1559.676067,classic_bike,member,202603-divvy-tripdata.csv
3773765,A679963188E41606,2026-03-07 14:40:43.466,2026-03-08 16:40:22.698,1559.653867,classic_bike,casual,202603-divvy-tripdata.csv
3894830,FBBF2C15825C4E6F,2026-03-07 12:28:27.947,2026-03-08 14:28:06.578,1559.643850,classic_bike,casual,202603-divvy-tripdata.csv


In [32]:
df_clean[
    [
        "ride_id",
        "started_at",
        "ended_at",
        "ride_length_minutes",
        "rideable_type",
        "member_casual",
        "source_file"
    ]
].sort_values(
    "ride_length_minutes"
).head(20)

,ride_id,started_at,ended_at,ride_length_minutes,rideable_type,member_casual,source_file
3155427,5740B9AA6176D32C,2025-11-02 01:59:56.801,2025-11-02 01:05:09.113,-54.794800,classic_bike,member,202511-divvy-tripdata.csv
3126068,ED8C9D500F271C3D,2025-11-02 01:57:24.553,2025-11-02 01:02:44.165,-54.673133,classic_bike,member,202511-divvy-tripdata.csv
3073056,222A5E8650C55311,2025-11-02 01:55:51.266,2025-11-02 01:01:50.026,-54.020667,electric_bike,member,202511-divvy-tripdata.csv
3041930,985A3462DAACFFCF,2025-11-02 01:59:07.835,2025-11-02 01:05:19.214,-53.810350,classic_bike,member,202511-divvy-tripdata.csv
3257358,E37AEB3E9CE2343E,2025-11-02 01:56:26.465,2025-11-02 01:03:08.183,-53.304700,electric_bike,member,202511-divvy-tripdata.csv
3010919,4A659F1BB40EDBF7,2025-11-02 01:55:38.177,2025-11-02 01:04:35.315,-51.047700,electric_bike,casual,202511-divvy-tripdata.csv
3061853,434E3D11ABA96891,2025-11-02 01:59:40.156,2025-11-02 01:08:51.303,-50.814217,electric_bike,casual,202511-divvy-tripdata.csv
3127426,4976DE4C5F1AA516,2025-11-02 01:56:44.152,2025-11-02 01:06:51.164,-49.883133,electric_bike,member,202511-divvy-tripdata.csv
3081824,83995E751A0DB5D5,2025-11-02 01:50:50.530,2025-11-02 01:01:15.026,-49.591733,electric_bike,member,202511-divvy-tripdata.csv
3178243,093B3FC8E465DA4A,2025-11-02 01:52:58.523,2025-11-02 01:04:31.464,-48.450983,electric_bike,member,202511-divvy-tripdata.csv


In [33]:
import pandas as pd, glob, os, numpy as np, pathlib, gc, math
files = sorted(glob.glob('/mnt/data/*divvy-tripdata.csv'))
len(files), [os.path.basename(f) for f in files]


(0, [])

In [34]:
import pandas as pd, glob, os
files = sorted(glob.glob('/mnt/data/*divvy-tripdata.csv'))
print(len(files))
print([os.path.basename(f) for f in files])


0
[]


In [42]:
parts=[]
for f in files:
    d=pd.read_csv(f,usecols=['ride_id','started_at','ended_at'])
    d['started_at']=pd.to_datetime(d['started_at'], errors='coerce')
    d['ended_at']=pd.to_datetime(d['ended_at'], errors='coerce')
    parts.append(d)
df=pd.concat(parts, ignore_index=True)
print(len(df), df['ride_id'].nunique())
mask=(df['started_at']>=pd.Timestamp('2025-07-01'))&(df['started_at']<pd.Timestamp('2026-07-01'))
d=df.loc[mask].drop_duplicates('ride_id', keep='first').copy()
d['ride_length_minutes']=(d['ended_at']-d['started_at']).dt.total_seconds()/60
s=d['ride_length_minutes']
print("rows",len(d))
print(s.describe(percentiles=[.001,.01,.25,.5,.75,.99,.999]))
for label, cond in [
    ('<=0', s<=0),('<1',s<1),('>60',s>60),('>1440',s>1440),('na',s.isna())]:
    print(label, int(cond.sum()))
print("top", d.nlargest(10,'ride_length_minutes')[['ride_id','started_at','ended_at','ride_length_minutes']].to_string(index=False))
print("bottom", d.nsmallest(10,'ride_length_minutes')[['ride_id','started_at','ended_at','ride_length_minutes']].to_string(index=False))


ValueError: No objects to concatenate

In [36]:
from pathlib import Path
import pandas as pd

cwd = Path.cwd()

if (cwd / "03_cleaned_data").exists():
    project_path = cwd
elif (cwd.parent / "03_cleaned_data").exists():
    project_path = cwd.parent
else:
    raise FileNotFoundError(
        "Could not locate the Cyclistic project folder."
    )

combined_file = (
    project_path
    / "03_cleaned_data"
    / "intermediate"
    / "cyclistic_202507_202606_combined_raw.parquet"
)

print("Loading:")
print(combined_file)

print("\nFile exists:", combined_file.exists())

df = pd.read_parquet(combined_file)

print("\nDataset loaded successfully.")
print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]}")

Loading:
c:\Users\OLUWATOSIN OLUWASEUN\Downloads\Cyclistic_Bike_Share_Case_Study\03_cleaned_data\intermediate\cyclistic_202507_202606_combined_raw.parquet

File exists: True

Dataset loaded successfully.
Rows: 5,932,349
Columns: 14


In [37]:
study_start = pd.Timestamp("2025-07-01")
study_end = pd.Timestamp("2026-07-01")

in_study_period = (
    (df["started_at"] >= study_start) &
    (df["started_at"] < study_end)
)

df_clean = df.loc[in_study_period].copy()

print(f"Rows after study-period filter: {len(df_clean):,}")

Rows after study-period filter: 5,932,247


In [38]:
rows_before = len(df_clean)

df_clean = (
    df_clean
    .drop_duplicates(
        subset="ride_id",
        keep="first"
    )
    .copy()
)

print(
    f"Duplicate rows removed: "
    f"{rows_before - len(df_clean):,}"
)

print(
    f"Rows remaining: "
    f"{len(df_clean):,}"
)

Duplicate rows removed: 35
Rows remaining: 5,932,212


In [39]:
df_clean["ride_length_minutes"] = (
    df_clean["ended_at"] - df_clean["started_at"]
).dt.total_seconds() / 60

print(df_clean["ride_length_minutes"].describe())

count    5.932212e+06
mean     1.576184e+01
std      5.337470e+01
min     -5.479480e+01
25%      5.359183e+00
50%      9.368150e+00
75%      1.647012e+01
max      1.559950e+03
Name: ride_length_minutes, dtype: float64


In [41]:
import pandas as pd
from pathlib import Path

# ============================================================
# CYCLISTIC COMBINED DATASET VALIDATION
# ============================================================

# 1. Locate project folder
cwd = Path.cwd()

if (cwd / "03_cleaned_data").exists():
    project_path = cwd
elif (cwd.parent / "03_cleaned_data").exists():
    project_path = cwd.parent
else:
    raise FileNotFoundError(
        "Could not locate the Cyclistic project folder."
    )

# 2. Locate combined Parquet file
combined_file = (
    project_path
    / "03_cleaned_data"
    / "intermediate"
    / "cyclistic_202507_202606_combined_raw.parquet"
)

if not combined_file.exists():
    raise FileNotFoundError(
        f"Combined Parquet file not found:\n{combined_file}"
    )

# 3. Load dataset
df = pd.read_parquet(combined_file)

# ============================================================
# EXPECTED VALUES
# ============================================================

expected_raw_rows = 5_932_349
expected_filtered_rows = 5_932_247
expected_final_rows = 5_932_212

expected_columns = [
    "ride_id",
    "rideable_type",
    "started_at",
    "ended_at",
    "start_station_name",
    "start_station_id",
    "end_station_name",
    "end_station_id",
    "start_lat",
    "start_lng",
    "end_lat",
    "end_lng",
    "member_casual",
    "source_file"
]

# ============================================================
# 4. VALIDATE RAW COMBINED DATASET
# ============================================================

assert len(df) == expected_raw_rows, (
    f"Unexpected raw row count. "
    f"Expected {expected_raw_rows:,}, found {len(df):,}."
)

assert df.columns.tolist() == expected_columns, (
    "Dataset columns do not match the expected 14-column structure."
)

assert df["started_at"].isna().sum() == 0, (
    "Invalid or missing started_at values detected."
)

assert df["ended_at"].isna().sum() == 0, (
    "Invalid or missing ended_at values detected."
)

# ============================================================
# 5. APPLY STUDY-PERIOD FILTER
# July 1, 2025 through June 30, 2026
# ============================================================

study_start = pd.Timestamp("2025-07-01")
study_end = pd.Timestamp("2026-07-01")

study_mask = (
    (df["started_at"] >= study_start)
    & (df["started_at"] < study_end)
)

df_clean = df.loc[study_mask].copy()

rows_outside_period = len(df) - len(df_clean)

assert len(df_clean) == expected_filtered_rows, (
    f"Unexpected row count after study-period filtering. "
    f"Expected {expected_filtered_rows:,}, "
    f"found {len(df_clean):,}."
)

# ============================================================
# 6. REMOVE DUPLICATE RIDE IDs
# ============================================================

rows_before_deduplication = len(df_clean)

df_clean = (
    df_clean
    .drop_duplicates(
        subset="ride_id",
        keep="first"
    )
    .copy()
)

duplicates_removed = (
    rows_before_deduplication - len(df_clean)
)

assert len(df_clean) == expected_final_rows, (
    f"Unexpected final row count. "
    f"Expected {expected_final_rows:,}, "
    f"found {len(df_clean):,}."
)

assert df_clean["ride_id"].duplicated().sum() == 0, (
    "Duplicate ride IDs remain after deduplication."
)

assert df_clean["ride_id"].nunique() == len(df_clean), (
    "ride_id is not unique in the cleaned dataframe."
)

# ============================================================
# 7. DISPLAY VALIDATION RESULTS
# ============================================================

print("=" * 65)
print("CYCLISTIC DATASET VALIDATION — PASSED")
print("=" * 65)

print(f"Parquet file:              {combined_file.name}")
print(f"Raw rows:                  {len(df):,}")
print(f"Expected columns:          {len(expected_columns)}")
print(f"Actual columns:            {df.shape[1]}")
print(f"Rows outside study period: {rows_outside_period:,}")
print(f"Rows after period filter:  {rows_before_deduplication:,}")
print(f"Duplicate rows removed:    {duplicates_removed:,}")
print(f"Final rows:                {len(df_clean):,}")
print(f"Unique ride IDs:           {df_clean['ride_id'].nunique():,}")
print(f"Remaining duplicates:      {df_clean['ride_id'].duplicated().sum():,}")
print(f"Missing started_at:        {df_clean['started_at'].isna().sum():,}")
print(f"Missing ended_at:          {df_clean['ended_at'].isna().sum():,}")

print("\nStudy period:")
print(
    f"{df_clean['started_at'].min()} "
    f"to {df_clean['started_at'].max()}"
)

print("\n✓ Combined dataset structure validated")
print("✓ Study-period filter validated")
print("✓ Duplicate removal validated")
print("✓ df_clean is ready for the next cleaning stage")

CYCLISTIC DATASET VALIDATION — PASSED
Parquet file:              cyclistic_202507_202606_combined_raw.parquet
Raw rows:                  5,932,349
Expected columns:          14
Actual columns:            14
Rows outside study period: 102
Rows after period filter:  5,932,247
Duplicate rows removed:    35
Final rows:                5,932,212
Unique ride IDs:           5,932,212
Remaining duplicates:      0
Missing started_at:        0
Missing ended_at:          0

Study period:
2025-07-01 00:00:21.054000 to 2026-06-30 23:58:47.113000

✓ Combined dataset structure validated
✓ Study-period filter validated
✓ Duplicate removal validated
✓ df_clean is ready for the next cleaning stage


## Ride Duration Validation

Ride duration is calculated as the difference between `ended_at` and `started_at`.

Before applying any filtering rules, the distribution of ride durations is examined to identify zero, negative, unusually short, and unusually long rides.

In [43]:
df_clean["ride_length_minutes"] = (
    df_clean["ended_at"] - df_clean["started_at"]
).dt.total_seconds() / 60

print("ride_length_minutes created successfully.")

df_clean[
    [
        "ride_id",
        "started_at",
        "ended_at",
        "ride_length_minutes"
    ]
].head()

ride_length_minutes created successfully.


,ride_id,started_at,ended_at,ride_length_minutes
0,455D43BD91D73437,2025-07-05 17:15:08.456,2025-07-05 17:25:47.079,10.643717
1,9D4A6B723ECD98CA,2025-07-01 13:57:38.878,2025-07-01 14:06:35.780,8.948367
2,C57044CF523302ED,2025-07-31 16:49:28.142,2025-07-31 17:15:28.999,26.014283
3,AFD35552E6685B6E,2025-07-17 09:36:21.058,2025-07-17 09:46:54.706,10.560800
4,C0582EBAA6CED519,2025-07-02 18:43:45.213,2025-07-02 18:57:06.687,13.357900


In [44]:
duration_summary = (
    df_clean["ride_length_minutes"]
    .describe(
        percentiles=[
            0.001,
            0.01,
            0.05,
            0.25,
            0.50,
            0.75,
            0.95,
            0.99,
            0.999
        ]
    )
)

duration_summary

count    5.932212e+06
mean     1.576184e+01
std      5.337470e+01
min     -5.479480e+01
0.1%     2.548333e-02
1%       2.414833e-01
5%       2.037959e+00
25%      5.359183e+00
50%      9.368150e+00
75%      1.647012e+01
95%      3.913337e+01
99%      8.709339e+01
99.9%    1.267057e+03
max      1.559950e+03
Name: ride_length_minutes, dtype: float64

In [45]:
duration_checks = pd.DataFrame({
    "condition": [
        "Duration <= 0 minutes",
        "Duration < 1 minute",
        "Duration > 60 minutes",
        "Duration > 12 hours",
        "Duration > 24 hours"
    ],
    "count": [
        (df_clean["ride_length_minutes"] <= 0).sum(),
        (df_clean["ride_length_minutes"] < 1).sum(),
        (df_clean["ride_length_minutes"] > 60).sum(),
        (df_clean["ride_length_minutes"] > 720).sum(),
        (df_clean["ride_length_minutes"] > 1440).sum()
    ]
})

duration_checks

,condition,count
0,Duration <= 0 minutes,29
1,Duration < 1 minute,162246
2,Duration > 60 minutes,122529
3,Duration > 12 hours,7902
4,Duration > 24 hours,5513


In [46]:
non_positive_rides = df_clean[
    df_clean["ride_length_minutes"] <= 0
].copy()

print(
    f"Rides with duration <= 0 minutes: "
    f"{len(non_positive_rides):,}"
)

non_positive_rides[
    [
        "ride_id",
        "started_at",
        "ended_at",
        "ride_length_minutes",
        "rideable_type",
        "member_casual",
        "source_file"
    ]
].sort_values(
    "ride_length_minutes"
).head(30)

Rides with duration <= 0 minutes: 29


,ride_id,started_at,ended_at,ride_length_minutes,rideable_type,member_casual,source_file
3155427,5740B9AA6176D32C,2025-11-02 01:59:56.801,2025-11-02 01:05:09.113,-54.794800,classic_bike,member,202511-divvy-tripdata.csv
3126068,ED8C9D500F271C3D,2025-11-02 01:57:24.553,2025-11-02 01:02:44.165,-54.673133,classic_bike,member,202511-divvy-tripdata.csv
3073056,222A5E8650C55311,2025-11-02 01:55:51.266,2025-11-02 01:01:50.026,-54.020667,electric_bike,member,202511-divvy-tripdata.csv
3041930,985A3462DAACFFCF,2025-11-02 01:59:07.835,2025-11-02 01:05:19.214,-53.810350,classic_bike,member,202511-divvy-tripdata.csv
3257358,E37AEB3E9CE2343E,2025-11-02 01:56:26.465,2025-11-02 01:03:08.183,-53.304700,electric_bike,member,202511-divvy-tripdata.csv
3010919,4A659F1BB40EDBF7,2025-11-02 01:55:38.177,2025-11-02 01:04:35.315,-51.047700,electric_bike,casual,202511-divvy-tripdata.csv
3061853,434E3D11ABA96891,2025-11-02 01:59:40.156,2025-11-02 01:08:51.303,-50.814217,electric_bike,casual,202511-divvy-tripdata.csv
3127426,4976DE4C5F1AA516,2025-11-02 01:56:44.152,2025-11-02 01:06:51.164,-49.883133,electric_bike,member,202511-divvy-tripdata.csv
3081824,83995E751A0DB5D5,2025-11-02 01:50:50.530,2025-11-02 01:01:15.026,-49.591733,electric_bike,member,202511-divvy-tripdata.csv
3178243,093B3FC8E465DA4A,2025-11-02 01:52:58.523,2025-11-02 01:04:31.464,-48.450983,electric_bike,member,202511-divvy-tripdata.csv


In [47]:
if len(non_positive_rides) > 0:
    non_positive_dates = (
        non_positive_rides["started_at"]
        .dt.date
        .value_counts()
        .sort_index()
    )
    
    display(non_positive_dates)
else:
    print("No non-positive ride durations found.")

started_at
2025-11-02    29
Name: count, dtype: int64

In [48]:
very_long_rides = df_clean[
    df_clean["ride_length_minutes"] > 1440
].copy()

print(
    f"Rides longer than 24 hours: "
    f"{len(very_long_rides):,}"
)

very_long_rides[
    [
        "ride_id",
        "started_at",
        "ended_at",
        "ride_length_minutes",
        "rideable_type",
        "member_casual",
        "source_file"
    ]
].sort_values(
    "ride_length_minutes",
    ascending=False
).head(30)

Rides longer than 24 hours: 5,513


,ride_id,started_at,ended_at,ride_length_minutes,rideable_type,member_casual,source_file
3818349,C82393BDD3070D5D,2026-03-07 16:07:09.988,2026-03-08 18:07:06.999,1559.950183,classic_bike,casual,202603-divvy-tripdata.csv
3895510,250556580DD308BE,2026-03-07 23:51:38.812,2026-03-09 01:51:33.033,1559.903683,classic_bike,member,202603-divvy-tripdata.csv
3892036,266FF1F3D3203623,2026-03-07 12:29:39.955,2026-03-08 14:29:33.587,1559.893867,classic_bike,casual,202603-divvy-tripdata.csv
3896432,F2DADD2100A676E7,2026-03-07 09:43:11.203,2026-03-08 11:43:04.158,1559.882583,classic_bike,casual,202603-divvy-tripdata.csv
3818481,EB04F9C062AAAF6A,2026-03-07 09:33:18.048,2026-03-08 11:33:09.975,1559.865450,classic_bike,casual,202603-divvy-tripdata.csv
3890261,7CFD1B6B5319BB28,2026-03-07 12:20:32.025,2026-03-08 14:20:23.251,1559.853767,classic_bike,casual,202603-divvy-tripdata.csv
3890264,B35EFB4539BB4B2B,2026-03-07 02:41:16.669,2026-03-08 04:41:03.312,1559.777383,classic_bike,casual,202603-divvy-tripdata.csv
3934526,13769591C639EB0C,2026-03-07 12:45:52.881,2026-03-08 14:45:33.445,1559.676067,classic_bike,member,202603-divvy-tripdata.csv
3773765,A679963188E41606,2026-03-07 14:40:43.466,2026-03-08 16:40:22.698,1559.653867,classic_bike,casual,202603-divvy-tripdata.csv
3894830,FBBF2C15825C4E6F,2026-03-07 12:28:27.947,2026-03-08 14:28:06.578,1559.643850,classic_bike,casual,202603-divvy-tripdata.csv


In [49]:
missing_summary = pd.DataFrame({
    "missing_count": df_clean.isna().sum(),
    "missing_percentage": (
        df_clean.isna().mean() * 100
    ).round(2)
})

missing_summary = (
    missing_summary
    .sort_values(
        "missing_count",
        ascending=False
    )
)

missing_summary

,missing_count,missing_percentage
end_station_name,1321632,22.28
end_station_id,1321632,22.28
start_station_name,1257495,21.20
start_station_id,1257495,21.20
end_lat,5581,0.09
end_lng,5581,0.09
started_at,0,0.00
ended_at,0,0.00
ride_id,0,0.00
rideable_type,0,0.00


In [50]:
required_fields = [
    "ride_id",
    "rideable_type",
    "started_at",
    "ended_at",
    "member_casual",
    "start_lat",
    "start_lng"
]

required_missing = (
    df_clean[required_fields]
    .isna()
    .sum()
)

required_missing

ride_id          0
rideable_type    0
started_at       0
ended_at         0
member_casual    0
start_lat        0
start_lng        0
dtype: int64

In [51]:
print("member_casual values:")
display(
    df_clean["member_casual"]
    .value_counts(dropna=False)
)

print("\nrideable_type values:")
display(
    df_clean["rideable_type"]
    .value_counts(dropna=False)
)

member_casual values:


member_casual
member    3815663
casual    2116549
Name: count, dtype: int64[pyarrow]


rideable_type values:


rideable_type
electric_bike    4012417
classic_bike     1919795
Name: count, dtype: int64[pyarrow]

In [52]:
coordinate_checks = pd.DataFrame({
    "condition": [
        "Invalid start latitude",
        "Invalid start longitude",
        "Invalid end latitude",
        "Invalid end longitude"
    ],
    "count": [
        (~df_clean["start_lat"].between(-90, 90)).sum(),
        (~df_clean["start_lng"].between(-180, 180)).sum(),
        (~df_clean["end_lat"].between(-90, 90)).sum(),
        (~df_clean["end_lng"].between(-180, 180)).sum()
    ]
})

coordinate_checks

,condition,count
0,Invalid start latitude,0
1,Invalid start longitude,0
2,Invalid end latitude,5581
3,Invalid end longitude,5581


In [53]:
df_clean["day_of_week"] = (
    df_clean["started_at"]
    .dt.day_name()
)

df_clean["day_of_week_num"] = (
    df_clean["started_at"]
    .dt.dayofweek
)

df_clean["month"] = (
    df_clean["started_at"]
    .dt.to_period("M")
    .astype(str)
)

df_clean["month_name"] = (
    df_clean["started_at"]
    .dt.month_name()
)

df_clean["month_num"] = (
    df_clean["started_at"]
    .dt.month
)

df_clean["start_hour"] = (
    df_clean["started_at"]
    .dt.hour
)

df_clean["is_weekend"] = (
    df_clean["day_of_week_num"] >= 5
)

print("Time-based features created.")

Time-based features created.


In [54]:
def assign_season(month):
    if month in [12, 1, 2]:
        return "Winter"
    elif month in [3, 4, 5]:
        return "Spring"
    elif month in [6, 7, 8]:
        return "Summer"
    else:
        return "Autumn"


df_clean["season"] = (
    df_clean["month_num"]
    .apply(assign_season)
)

df_clean[
    [
        "started_at",
        "day_of_week",
        "month",
        "start_hour",
        "is_weekend",
        "season",
        "ride_length_minutes"
    ]
].head()

,started_at,day_of_week,month,start_hour,is_weekend,season,ride_length_minutes
0,2025-07-05 17:15:08.456,Saturday,2025-07,17,True,Summer,10.643717
1,2025-07-01 13:57:38.878,Tuesday,2025-07,13,False,Summer,8.948367
2,2025-07-31 16:49:28.142,Thursday,2025-07,16,False,Summer,26.014283
3,2025-07-17 09:36:21.058,Thursday,2025-07,9,False,Summer,10.560800
4,2025-07-02 18:43:45.213,Wednesday,2025-07,18,False,Summer,13.357900


In [55]:
print("Days of week:")
print(
    sorted(
        df_clean["day_of_week"]
        .dropna()
        .unique()
    )
)

print("\nMonths:")
print(
    sorted(
        df_clean["month"]
        .dropna()
        .unique()
    )
)

print("\nSeasons:")
print(
    df_clean["season"]
    .value_counts()
)

Days of week:
['Friday', 'Monday', 'Saturday', 'Sunday', 'Thursday', 'Tuesday', 'Wednesday']

Months:
['2025-07', '2025-08', '2025-09', '2025-10', '2025-11', '2025-12', '2026-01', '2026-02', '2026-03', '2026-04', '2026-05', '2026-06']

Seasons:
season
Summer    2316247
Autumn    1717170
Spring    1419014
Winter     479781
Name: count, dtype: int64


In [56]:
print("=" * 60)
print("RIDE DURATION VALIDATION")
print("=" * 60)

print(
    f"Minimum ride length: "
    f"{df_clean['ride_length_minutes'].min():,.2f} minutes"
)

print(
    f"Median ride length: "
    f"{df_clean['ride_length_minutes'].median():,.2f} minutes"
)

print(
    f"99th percentile: "
    f"{df_clean['ride_length_minutes'].quantile(0.99):,.2f} minutes"
)

print(
    f"Maximum ride length: "
    f"{df_clean['ride_length_minutes'].max():,.2f} minutes"
)

print()

print(
    f"Duration <= 0 minutes: "
    f"{(df_clean['ride_length_minutes'] <= 0).sum():,}"
)

print(
    f"Duration < 1 minute: "
    f"{(df_clean['ride_length_minutes'] < 1).sum():,}"
)

print(
    f"Duration > 60 minutes: "
    f"{(df_clean['ride_length_minutes'] > 60).sum():,}"
)

print(
    f"Duration > 24 hours: "
    f"{(df_clean['ride_length_minutes'] > 1440).sum():,}"
)

print("\nDates containing rides <= 0 minutes:")

non_positive_dates = (
    df_clean.loc[
        df_clean["ride_length_minutes"] <= 0,
        "started_at"
    ]
    .dt.date
    .value_counts()
    .sort_index()
)

display(non_positive_dates)

RIDE DURATION VALIDATION
Minimum ride length: -54.79 minutes
Median ride length: 9.37 minutes
99th percentile: 87.09 minutes
Maximum ride length: 1,559.95 minutes

Duration <= 0 minutes: 29
Duration < 1 minute: 162,246
Duration > 60 minutes: 122,529
Duration > 24 hours: 5,513

Dates containing rides <= 0 minutes:


started_at
2025-11-02    29
Name: count, dtype: int64

## Final Ride-Duration Cleaning Rule

Following exploratory validation of ride duration:

- rides with durations less than or equal to zero minutes were excluded because the recorded end time does not represent a valid positive-duration trip;
- rides longer than 24 hours were excluded as extreme duration outliers that could materially distort summary statistics;
- positive rides shorter than one minute were retained because they represent a substantial number of observations and there was insufficient evidence to classify them as invalid.

The 24-hour threshold is an analytical cleaning decision rather than a rule specified by the source dataset.

Missing station information was retained because the primary analysis focuses on rider type, duration, time, day, month, and season and does not require complete station identifiers.

In [57]:
# ============================================================
# FINAL RIDE-DURATION CLEANING
# ============================================================

rows_before_duration_cleaning = len(df_clean)

non_positive_count = (
    df_clean["ride_length_minutes"] <= 0
).sum()

over_24h_count = (
    df_clean["ride_length_minutes"] > 1440
).sum()

positive_under_1min_count = (
    (df_clean["ride_length_minutes"] > 0) &
    (df_clean["ride_length_minutes"] < 1)
).sum()

print("Before final duration cleaning:")
print(f"Rows:                         {rows_before_duration_cleaning:,}")
print(f"Non-positive rides:           {non_positive_count:,}")
print(f"Rides longer than 24 hours:   {over_24h_count:,}")
print(f"Positive rides under 1 minute:{positive_under_1min_count:,}")


# Keep only positive rides of 24 hours or less
duration_mask = (
    (df_clean["ride_length_minutes"] > 0) &
    (df_clean["ride_length_minutes"] <= 1440)
)

df_final = df_clean.loc[duration_mask].copy()

rows_removed = (
    rows_before_duration_cleaning - len(df_final)
)

print("\nAfter final duration cleaning:")
print(f"Rows removed: {rows_removed:,}")
print(f"Rows retained:{len(df_final):,}")

Before final duration cleaning:
Rows:                         5,932,212
Non-positive rides:           29
Rides longer than 24 hours:   5,513
Positive rides under 1 minute:162,217

After final duration cleaning:
Rows removed: 5,542
Rows retained:5,926,670


In [58]:
# ============================================================
# FINAL DATA QUALITY CHECK
# ============================================================

expected_final_rows = 5_926_670

assert len(df_final) == expected_final_rows, (
    f"Expected {expected_final_rows:,} final rows, "
    f"but found {len(df_final):,}."
)

assert df_final["ride_id"].duplicated().sum() == 0, (
    "Duplicate ride IDs remain."
)

assert (df_final["ride_length_minutes"] <= 0).sum() == 0, (
    "Non-positive ride durations remain."
)

assert (df_final["ride_length_minutes"] > 1440).sum() == 0, (
    "Ride durations longer than 24 hours remain."
)

assert df_final["started_at"].isna().sum() == 0, (
    "Missing started_at values remain."
)

assert df_final["ended_at"].isna().sum() == 0, (
    "Missing ended_at values remain."
)

assert df_final["member_casual"].isna().sum() == 0, (
    "Missing member_casual values remain."
)

print("=" * 65)
print("FINAL CYCLISTIC DATA QUALITY CHECK — PASSED")
print("=" * 65)

print(f"Final rows:                 {len(df_final):,}")
print(f"Unique ride IDs:            {df_final['ride_id'].nunique():,}")
print(f"Duplicate ride IDs:         {df_final['ride_id'].duplicated().sum():,}")
print(f"Duration <= 0:              {(df_final['ride_length_minutes'] <= 0).sum():,}")
print(f"Duration > 24 hours:        {(df_final['ride_length_minutes'] > 1440).sum():,}")
print(f"Minimum duration:           {df_final['ride_length_minutes'].min():.2f} minutes")
print(f"Maximum duration:           {df_final['ride_length_minutes'].max():.2f} minutes")
print(f"Study months:               {df_final['month'].nunique()}")
print(f"Missing started_at:         {df_final['started_at'].isna().sum():,}")
print(f"Missing ended_at:           {df_final['ended_at'].isna().sum():,}")

FINAL CYCLISTIC DATA QUALITY CHECK — PASSED
Final rows:                 5,926,670
Unique ride IDs:            5,926,670
Duplicate ride IDs:         0
Duration <= 0:              0
Duration > 24 hours:        0
Minimum duration:           0.00 minutes
Maximum duration:           1439.98 minutes
Study months:               12
Missing started_at:         0
Missing ended_at:           0


In [59]:
# ============================================================
# SAVE FINAL CLEANED DATASET
# ============================================================

final_output = (
    project_path
    / "03_cleaned_data"
    / "cyclistic_202507_202606_cleaned.parquet"
)

df_final.to_parquet(
    final_output,
    index=False
)

print("Final cleaned dataset saved successfully.")
print("\nLocation:")
print(final_output)

print("\nFile exists:", final_output.exists())

if final_output.exists():
    file_size_mb = final_output.stat().st_size / (1024 ** 2)
    print(f"File size: {file_size_mb:,.2f} MB")

Final cleaned dataset saved successfully.

Location:
c:\Users\OLUWATOSIN OLUWASEUN\Downloads\Cyclistic_Bike_Share_Case_Study\03_cleaned_data\cyclistic_202507_202606_cleaned.parquet

File exists: True
File size: 292.59 MB


In [60]:
cleaning_summary = pd.DataFrame({
    "stage": [
        "Raw combined dataset",
        "Outside study period removed",
        "Duplicate ride IDs removed",
        "Non-positive durations removed",
        "Rides over 24 hours removed",
        "Final cleaned dataset"
    ],
    "records": [
        5_932_349,
        102,
        35,
        non_positive_count,
        over_24h_count,
        len(df_final)
    ]
})

cleaning_summary

,stage,records
0,Raw combined dataset,5932349
1,Outside study period removed,102
2,Duplicate ride IDs removed,35
3,Non-positive durations removed,29
4,Rides over 24 hours removed,5513
5,Final cleaned dataset,5926670


In [61]:
cleaning_summary_output = (
    project_path
    / "06_documentation"
    / "cleaning_summary.csv"
)

cleaning_summary.to_csv(
    cleaning_summary_output,
    index=False
)

print("Cleaning summary saved:")
print(cleaning_summary_output)

Cleaning summary saved:
c:\Users\OLUWATOSIN OLUWASEUN\Downloads\Cyclistic_Bike_Share_Case_Study\06_documentation\cleaning_summary.csv
